# Gene–Gene Interactions — Mouse KG Processing
**Project:** MetaboGlue / EvoAge KG &nbsp;|&nbsp; **Contributor:** Arushi

**Sources processed:**
- **File 1 (STRING):** `10090.protein.links.detailed.v12.0.txt` → Mouse Protein–Protein interactions, ENSMUSG IDs resolved to gene symbols via gProfiler + NCBI
- **File 2 (MouseNet v2):** `MouseNetV2_symbol.txt` → Mouse Gene–Gene interactions, gene-sequence filtered, annotated via NCBI

All inputs are read from `BASE_PATH` / `DB_BASE_PATH`.  
All outputs are written to `OUT_PATH`.

---



## 0. Path Configuration

In [33]:
import os

# ── Edit only these lines ────────────────────────────────────────────────────
BASE_PATH    = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"   # raw input data
DB_BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/"   # shared reference databases
OUT_PATH     = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/mmus/"   # all output CSVs land here

# Sub-paths (derived automatically)
MOUSE_DB_PATH      = os.path.join(BASE_PATH, "string/mmu/")
NCBI_DB_PATH       = os.path.join(DB_BASE_PATH, "ncbi")
STRING_PATH        = os.path.join(BASE_PATH, "string", "mmu")
MOUSENET_PATH      = os.path.join(BASE_PATH, "mousenet")
MOUSENETOUT_PATH = os.path.join("/storage/Arushi/090526_EvoAge/kg_formation/processed_data/mousenet/")
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_PATH, exist_ok=True)
print("BASE_PATH    :", BASE_PATH)
print("DB_BASE_PATH :", DB_BASE_PATH)
print("OUT_PATH     :", OUT_PATH)

BASE_PATH    : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
DB_BASE_PATH : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/
OUT_PATH     : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/mmus/


## 1. Imports

In [2]:
import os
import pickle
import numpy as np
import pandas as pd


## 2. Shared Helpers

In [3]:
# Canonical column order for all Gene–Gene KG files
KG_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name',
    'species',   # Bug 6 fix: was missing from required_columns
]

def align_cols(df):
    """Add any missing KG_COLS as NA and return df with only those columns in order."""
    for col in KG_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df[KG_COLS]

def merge_sources(x):
    """Combine kg_source values with '::' separator (used during dedup)."""
    return '::'.join(sorted(set(x.dropna().astype(str))))

def dedup(df, group_cols=('head', 'relation', 'tail')):
    """Deduplicate by (head, relation, tail); merge kg_source strings."""
    agg = {c: 'first' for c in KG_COLS if c not in group_cols and c != 'kg_source'}
    agg['kg_source'] = merge_sources
    return df.groupby(list(group_cols), as_index=False).agg(agg)


## 3. Reference Dictionaries

### 3.1 gProfiler: Mouse ENSMUSG → Gene Symbol & Description

In [8]:
GPRO = pd.read_csv(
    os.path.join(MOUSE_DB_PATH, "gProfiler_mmusculus_4-10-2025_12-28-24 AM.csv")
)
GPRO = GPRO[~GPRO['converted_alias'].isna()]

GPRO_dict      = dict(zip(GPRO['initial_alias'], GPRO['name']))         # ENSMUSG → symbol
GPRO_Desc_dict = dict(zip(GPRO['name'],          GPRO['description']))  # symbol → description
del GPRO
print(f"GPRO_dict      : {len(GPRO_dict):,} entries")
print(f"GPRO_Desc_dict : {len(GPRO_Desc_dict):,} entries")


GPRO_dict      : 21,175 entries
GPRO_Desc_dict : 21,155 entries


### 3.2 NCBI Mouse Gene Info (Symbol → Description / ENSMUSG)

In [27]:
NCBI_MOUSE_gene = pd.read_csv(
    os.path.join(NCBI_DB_PATH, "Mus_musculus.gene_info"), sep='\t'
)
NCBI_MOUSE_gene['ENSMBL'] = NCBI_MOUSE_gene['dbXrefs'].str.extract(
    r'(ENSMUSG\d+)', expand=False
)

NCBI_Mouse_gene_Locus_2_GeneSymd_ict = dict(
    zip(NCBI_MOUSE_gene['ENSMBL'],  NCBI_MOUSE_gene['Symbol'])
)
NCBI_Mouse_SYM_Descrip_dict = dict(
    zip(NCBI_MOUSE_gene['Symbol'], NCBI_MOUSE_gene['description'])
)
del NCBI_MOUSE_gene
print(f"NCBI_Mouse_gene_Locus_2_GeneSymd_ict : {len(NCBI_Mouse_gene_Locus_2_GeneSymd_ict):,}")
print(f"NCBI_Mouse_SYM_Descrip_dict          : {len(NCBI_Mouse_SYM_Descrip_dict):,}")


NCBI_Mouse_gene_Locus_2_GeneSymd_ict : 34,765
NCBI_Mouse_SYM_Descrip_dict          : 112,035


### 3.3 Mouse Gene → Sequence (for MouseNet v2 sequence filtering)

In [12]:
# Gene_Seq = pd.read_csv(
#     os.path.join(MOUSE_GG_DATA_PATH, "MOUSE_ALL_GENE_SEQUENCE_ENS_DATABASE.csv")
# )
# symbol_dict_Mouse_G_2_S = dict(zip(Gene_Seq['Gene_sym'], Gene_Seq['SEQ']))
# del Gene_Seq
# print(f"symbol_dict_Mouse_G_2_S : {len(symbol_dict_Mouse_G_2_S):,} entries")


## 4. File 1 — STRING Mouse Protein–Protein Interactions
**Source:** `10090.protein.links.detailed.v12.0.txt`  
ENSMUSG protein IDs resolved to gene symbols via gProfiler dict.
> **Bug 4 fix:** Standardised column naming — raw ENSMUSG IDs in `Head`/`Tail`; mapped gene symbols in `head`/`tail`.  
> **Bug 5 fix:** `sep='\s'` → `sep='\s+'` with `engine='python'` to handle multi-space delimiters correctly.


In [13]:
STRING_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/data_collection/string/mmu'

In [14]:
# Bug 5 fix: sep='\s+' with engine='python' for whitespace-separated files
C_P_P = pd.read_csv(
    os.path.join(STRING_PATH, "10090.protein.links.detailed.v12.0.txt"),
    sep='\s+', engine='python'
)
C_P_P = C_P_P.rename(columns={'protein1': 'Head', 'protein2': 'Tail'})

# Strip species prefix
C_P_P['Head'] = C_P_P['Head'].str.replace('10090.', '', regex=False)
C_P_P['Tail'] = C_P_P['Tail'].str.replace('10090.', '', regex=False)

print(f"Raw STRING rows: {len(C_P_P):,}")
C_P_P.head()

Raw STRING rows: 12,684,354


,Head,Tail,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
0,ENSMUSP00000000001,ENSMUSP00000027991,0,0,0,56,594,500,492,889
1,ENSMUSP00000000001,ENSMUSP00000137332,0,0,0,0,0,0,163,163
2,ENSMUSP00000000001,ENSMUSP00000041756,0,0,0,110,117,0,65,201
3,ENSMUSP00000000001,ENSMUSP00000075170,0,0,0,0,604,900,301,969
4,ENSMUSP00000000001,ENSMUSP00000110978,0,0,0,48,228,0,84,267


In [15]:
# Map ENSMUSG → gene symbol (Bug 4 fix: lowercase mapped columns, uppercase raw IDs)
C_P_P['head'] = C_P_P['Head'].map(GPRO_dict)
C_P_P['tail'] = C_P_P['Tail'].map(GPRO_dict)
C_P_P

,Head,Tail,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,head,tail
0,ENSMUSP00000000001,ENSMUSP00000027991,0,0,0,56,594,500,492,889,Gnai3,Rgs4
1,ENSMUSP00000000001,ENSMUSP00000137332,0,0,0,0,0,0,163,163,Gnai3,Cmtm4
2,ENSMUSP00000000001,ENSMUSP00000041756,0,0,0,110,117,0,65,201,Gnai3,Arl5a
3,ENSMUSP00000000001,ENSMUSP00000075170,0,0,0,0,604,900,301,969,Gnai3,Drd2
4,ENSMUSP00000000001,ENSMUSP00000110978,0,0,0,48,228,0,84,267,Gnai3,Grm8
...,...,...,...,...,...,...,...,...,...,...,...,...
12684349,ENSMUSP00000159257,ENSMUSP00000047904,0,0,0,0,0,0,197,197,Carlr,Cmc1
12684350,ENSMUSP00000159257,ENSMUSP00000062110,0,0,0,0,0,0,367,367,Carlr,Tapt1
12684351,ENSMUSP00000159257,ENSMUSP00000099018,0,0,0,0,0,0,170,169,Carlr,Peli1
12684352,ENSMUSP00000159257,ENSMUSP00000023468,0,0,0,0,0,0,253,252,Carlr,Spag6l


In [16]:
# Annotate descriptions
C_P_P['head_detail_name'] = C_P_P['head'].map(GPRO_Desc_dict)
C_P_P['tail_detail_name'] = C_P_P['tail'].map(GPRO_Desc_dict)

# Drop rows without gene-level annotation
C_P_P = C_P_P[~C_P_P['head_detail_name'].isna()]
C_P_P = C_P_P[~C_P_P['tail_detail_name'].isna()]

print(f"After symbol + description filter: {len(C_P_P):,}")


After symbol + description filter: 12,426,958


In [17]:
# Set KG metadata
C_P_P['relation']      = 'Gene_Gene'
C_P_P['head_type']     = 'Gene'
C_P_P['tail_type']     = 'Gene'
C_P_P['relation_type'] = np.nan
C_P_P['kg_source']     = 'STRING'
C_P_P['head_id_is']    = 'NCBI_ID'
C_P_P['tail_id_is']    = 'NCBI_ID'
C_P_P['species']       = 'M.musculus'

# Dedup within this source
C_P_P_aligned = align_cols(C_P_P)
C_P_P_dedup   = dedup(C_P_P_aligned)

print(f"After dedup: {len(C_P_P_dedup):,}")
C_P_P_dedup.head()

After dedup: 12,418,376


,head,relation,tail,head_type,relation_type,tail_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species,kg_source
0,0610010K14Rik,Gene_Gene,Actr5,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 0610010K14 gene [Source:MGI Symbol;...,ARP5 actin-related protein 5 [Source:MGI Symbo...,M.musculus,STRING
1,0610010K14Rik,Gene_Gene,Actr8,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 0610010K14 gene [Source:MGI Symbol;...,ARP8 actin-related protein 8 [Source:MGI Symbo...,M.musculus,STRING
2,0610010K14Rik,Gene_Gene,Ak6,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 0610010K14 gene [Source:MGI Symbol;...,adenylate kinase 6 [Source:MGI Symbol;Acc:MGI:...,M.musculus,STRING
3,0610010K14Rik,Gene_Gene,Akap8,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 0610010K14 gene [Source:MGI Symbol;...,A kinase anchor protein 8 [Source:MGI Symbol;A...,M.musculus,STRING
4,0610010K14Rik,Gene_Gene,Ap2s1,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 0610010K14 gene [Source:MGI Symbol;...,"adaptor-related protein complex 2, sigma 1 sub...",M.musculus,STRING


In [18]:
C_P_P_dedup

,head,relation,tail,head_type,relation_type,tail_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species,kg_source
0,0610010K14Rik,Gene_Gene,Actr5,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 0610010K14 gene [Source:MGI Symbol;...,ARP5 actin-related protein 5 [Source:MGI Symbo...,M.musculus,STRING
1,0610010K14Rik,Gene_Gene,Actr8,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 0610010K14 gene [Source:MGI Symbol;...,ARP8 actin-related protein 8 [Source:MGI Symbo...,M.musculus,STRING
2,0610010K14Rik,Gene_Gene,Ak6,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 0610010K14 gene [Source:MGI Symbol;...,adenylate kinase 6 [Source:MGI Symbol;Acc:MGI:...,M.musculus,STRING
3,0610010K14Rik,Gene_Gene,Akap8,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 0610010K14 gene [Source:MGI Symbol;...,A kinase anchor protein 8 [Source:MGI Symbol;A...,M.musculus,STRING
4,0610010K14Rik,Gene_Gene,Ap2s1,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 0610010K14 gene [Source:MGI Symbol;...,"adaptor-related protein complex 2, sigma 1 sub...",M.musculus,STRING
...,...,...,...,...,...,...,...,...,...,...,...,...
12418371,mt-Nd6,Gene_Gene,mt-Nd2,Gene,NaN,Gene,NCBI_ID,NCBI_ID,mitochondrially encoded NADH dehydrogenase 6 [...,mitochondrially encoded NADH dehydrogenase 2 [...,M.musculus,STRING
12418372,mt-Nd6,Gene_Gene,mt-Nd3,Gene,NaN,Gene,NCBI_ID,NCBI_ID,mitochondrially encoded NADH dehydrogenase 6 [...,mitochondrially encoded NADH dehydrogenase 3 [...,M.musculus,STRING
12418373,mt-Nd6,Gene_Gene,mt-Nd4,Gene,NaN,Gene,NCBI_ID,NCBI_ID,mitochondrially encoded NADH dehydrogenase 6 [...,mitochondrially encoded NADH dehydrogenase 4 [...,M.musculus,STRING
12418374,mt-Nd6,Gene_Gene,mt-Nd4l,Gene,NaN,Gene,NCBI_ID,NCBI_ID,mitochondrially encoded NADH dehydrogenase 6 [...,mitochondrially encoded NADH dehydrogenase 4L ...,M.musculus,STRING


In [21]:
C_P_P_dedup.to_csv('/storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/mmus/string_MOUSE_GENE_GENE.csv',index = None)

## 5. File 2 — MouseNet v2 Gene–Gene Interactions
**Source:** `MouseNetV2_symbol.txt`  


In [28]:
Mus_M_G_G = pd.read_csv(
    os.path.join(MOUSENET_PATH, "MouseNetV2_symbol.txt"), sep='\t'
)
Mus_M_G_G = Mus_M_G_G.rename(columns={
    'GeneA': 'Head',
    'GeneB': 'Tail',
    'Score': 'combined_score',
})
Mus_M_G_G['Relation']        = 'Gene_Gene'
Mus_M_G_G['Relation_Source'] = 'Mouse_Net_v2'
Mus_M_G_G['head_type']       = 'Gene'
Mus_M_G_G['tail_type']       = 'Gene'

desired_columns = ['Head', 'Relation', 'Tail', 'combined_score',
                   'head_type', 'tail_type', 'Relation_Source']
Mus_M_G_G = Mus_M_G_G[desired_columns]

print(f"Raw MouseNet rows: {len(Mus_M_G_G):,}")
unique_genes = set(Mus_M_G_G['Head'].tolist() + Mus_M_G_G['Tail'].tolist())
print(f"Unique genes: {len(unique_genes):,}")
Mus_M_G_G


Raw MouseNet rows: 788,080
Unique genes: 17,714


,Head,Relation,Tail,combined_score,head_type,tail_type,Relation_Source
0,S100a8,Gene_Gene,S100a9,5.360552,Gene,Gene,Mouse_Net_v2
1,Rps28,Gene_Gene,Rpl38,5.325248,Gene,Gene,Mouse_Net_v2
2,Cox6b1,Gene_Gene,Cox7a2,5.299989,Gene,Gene,Mouse_Net_v2
3,Atp5a1,Gene_Gene,Atp5d,5.265136,Gene,Gene,Mouse_Net_v2
4,Rpl37a,Gene_Gene,Rpl38,5.259759,Gene,Gene,Mouse_Net_v2
...,...,...,...,...,...,...,...
788075,Slc26a1,Gene_Gene,Slc7a15,0.741315,Gene,Gene,Mouse_Net_v2
788076,Adrb3,Gene_Gene,Ppa1,0.741313,Gene,Gene,Mouse_Net_v2
788077,Gsta2,Gene_Gene,Sgk2,0.741310,Gene,Gene,Mouse_Net_v2
788078,Has1,Gene_Gene,Abca8b,0.741308,Gene,Gene,Mouse_Net_v2


In [29]:
# Reload, drop SEQ columns, and set KG metadata

Mus_M_G_G.columns = Mus_M_G_G.columns.str.lower()   # normalise all to lowercase

Mus_M_G_G['relation']      = 'Gene_Gene'
Mus_M_G_G['head_type']     = 'Gene'
Mus_M_G_G['tail_type']     = 'Gene'
Mus_M_G_G['relation_type'] = np.nan
Mus_M_G_G['kg_source']     = Mus_M_G_G['relation_source']   # preserve original source label
Mus_M_G_G['head_id_is']    = 'NCBI_ID'
Mus_M_G_G['tail_id_is']    = 'NCBI_ID'
Mus_M_G_G['species']       = 'M.musculus'

Mus_M_G_G['head_detail_name'] = Mus_M_G_G['head'].map(NCBI_Mouse_SYM_Descrip_dict)
Mus_M_G_G['tail_detail_name'] = Mus_M_G_G['tail'].map(NCBI_Mouse_SYM_Descrip_dict)

Mus_M_G_G = Mus_M_G_G[~Mus_M_G_G['head_detail_name'].isna()]
Mus_M_G_G = Mus_M_G_G[~Mus_M_G_G['tail_detail_name'].isna()]

print(f"MouseNet v2 rows after annotation filter: {len(Mus_M_G_G):,}")
Mus_M_G_G


MouseNet v2 rows after annotation filter: 698,626


,head,relation,tail,combined_score,head_type,tail_type,relation_source,relation_type,kg_source,head_id_is,tail_id_is,species,head_detail_name,tail_detail_name
0,S100a8,Gene_Gene,S100a9,5.360552,Gene,Gene,Mouse_Net_v2,NaN,Mouse_Net_v2,NCBI_ID,NCBI_ID,M.musculus,S100 calcium binding protein A8 (calgranulin A),S100 calcium binding protein A9 (calgranulin B)
1,Rps28,Gene_Gene,Rpl38,5.325248,Gene,Gene,Mouse_Net_v2,NaN,Mouse_Net_v2,NCBI_ID,NCBI_ID,M.musculus,ribosomal protein S28,ribosomal protein L38
2,Cox6b1,Gene_Gene,Cox7a2,5.299989,Gene,Gene,Mouse_Net_v2,NaN,Mouse_Net_v2,NCBI_ID,NCBI_ID,M.musculus,"cytochrome c oxidase, subunit 6B1",cytochrome c oxidase subunit 7A2
4,Rpl37a,Gene_Gene,Rpl38,5.259759,Gene,Gene,Mouse_Net_v2,NaN,Mouse_Net_v2,NCBI_ID,NCBI_ID,M.musculus,ribosomal protein L37a,ribosomal protein L38
5,Col1a1,Gene_Gene,Col1a2,5.245888,Gene,Gene,Mouse_Net_v2,NaN,Mouse_Net_v2,NCBI_ID,NCBI_ID,M.musculus,"collagen, type I, alpha 1","collagen, type I, alpha 2"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788075,Slc26a1,Gene_Gene,Slc7a15,0.741315,Gene,Gene,Mouse_Net_v2,NaN,Mouse_Net_v2,NCBI_ID,NCBI_ID,M.musculus,solute carrier family 26 (sulfate transporter)...,solute carrier family 7 (cationic amino acid t...
788076,Adrb3,Gene_Gene,Ppa1,0.741313,Gene,Gene,Mouse_Net_v2,NaN,Mouse_Net_v2,NCBI_ID,NCBI_ID,M.musculus,"adrenergic receptor, beta 3",pyrophosphatase (inorganic) 1
788077,Gsta2,Gene_Gene,Sgk2,0.741310,Gene,Gene,Mouse_Net_v2,NaN,Mouse_Net_v2,NCBI_ID,NCBI_ID,M.musculus,"glutathione S-transferase, alpha 2 (Yc2)",serum/glucocorticoid regulated kinase 2
788078,Has1,Gene_Gene,Abca8b,0.741308,Gene,Gene,Mouse_Net_v2,NaN,Mouse_Net_v2,NCBI_ID,NCBI_ID,M.musculus,hyaluronan synthase 1,"ATP-binding cassette, sub-family A member 8b"


In [31]:
Mus_M_G_G = align_cols(Mus_M_G_G)
Mus_M_G_G   = dedup(Mus_M_G_G)
Mus_M_G_G


,head,relation,tail,head_type,relation_type,tail_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species,kg_source
0,1110004F10Rik,Gene_Gene,Aasdhppt,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 1110004F10 gene,aminoadipate-semialdehyde dehydrogenase-phosph...,M.musculus,Mouse_Net_v2
1,1110004F10Rik,Gene_Gene,Ankrd17,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 1110004F10 gene,ankyrin repeat domain 17,M.musculus,Mouse_Net_v2
2,1110004F10Rik,Gene_Gene,Bzw2,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 1110004F10 gene,basic leucine zipper and W2 domains 2,M.musculus,Mouse_Net_v2
3,1110004F10Rik,Gene_Gene,Chordc1,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 1110004F10 gene,cysteine and histidine rich domain containing 1,M.musculus,Mouse_Net_v2
4,1110004F10Rik,Gene_Gene,Cmtr1,Gene,NaN,Gene,NCBI_ID,NCBI_ID,RIKEN cDNA 1110004F10 gene,cap methyltransferase 1,M.musculus,Mouse_Net_v2
...,...,...,...,...,...,...,...,...,...,...,...,...
698621,Zzz3,Gene_Gene,Yeats2,Gene,NaN,Gene,NCBI_ID,NCBI_ID,"zinc finger, ZZ domain containing 3",YEATS domain containing 2,M.musculus,Mouse_Net_v2
698622,Zzz3,Gene_Gene,Zbtb9,Gene,NaN,Gene,NCBI_ID,NCBI_ID,"zinc finger, ZZ domain containing 3",zinc finger and BTB domain containing 9,M.musculus,Mouse_Net_v2
698623,Zzz3,Gene_Gene,Zfp212,Gene,NaN,Gene,NCBI_ID,NCBI_ID,"zinc finger, ZZ domain containing 3",Zinc finger protein 212,M.musculus,Mouse_Net_v2
698624,Zzz3,Gene_Gene,Zfp36l2,Gene,NaN,Gene,NCBI_ID,NCBI_ID,"zinc finger, ZZ domain containing 3","zinc finger protein 36, C3H type-like 2",M.musculus,Mouse_Net_v2


In [32]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/mmus/'

In [34]:
print(f"After dedup: {len(Mus_M_G_G):,}")
Mus_M_G_G.to_csv(os.path.join(MOUSENETOUT_PATH, "mousenet_MOUSE_GENE_GENE.csv"), index=False)



After dedup: 698,626
